# pysent - quickstart

**pysent** turns Sentinel-1 and Sentinel-2 SAFE products into display-ready
GeoTIFF quicklooks. It is the processing core of the NBS ingestion service,
packaged as a library so the service, these notebooks and any other tooling all
run the *same* code.

A quicklook is produced in three steps, and the library exposes each one:

| Step | Sentinel-1 | Sentinel-2 |
|---|---|---|
| **warp** to a target CRS/resolution | `_warp_sentinel_s1_safe_amplitude` (GCP/TPS) | `_warp_sentinel_s2_rgb` (stacked band VRT) |
| **stretch** to 8-bit | `stretch_sentinel_s1_grayscale` (percentile → gray + alpha) | `stretch_sentinel_s2_rgb` (per-band percentile → RGB) |
| **write** tiled, compressed, with overviews | `process_sentinel_s1_safe` | `process_sentinel_s2_safe` |

The `process_*` entry points do all three; the individual functions let you
inspect or tune a single stage, which is what these notebooks do.

> **Running this**: with no parameters set it uses the small committed test
> fixtures, so it works anywhere. Set `DATA_ROOT` to a mounted archive (or
> `SAFE_PATH` / `IDENTIFIER`) to run the full pipeline on a real product.

In [ ]:
# --- papermill parameters -------------------------------------------------
# Leave everything as-is to run against the committed test fixtures (what CI
# does). Point any of these at real data for the full pipeline.
DATA_ROOT = ""      # mounted NBS archive, e.g. "/data/nbsArchive"
SAFE_PATH = ""      # explicit .SAFE / .zip product
IDENTIFIER = ""     # catalogue UUID, resolved via pysent.archive
ENDPOINT = None     # None -> NBS_SENTINEL_CSW_ENDPOINT / https://nbs.csw.met.no
OUTPUT_DIR = "_output"
PLATFORM = "S2"

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import rasterio

import pysent
import nbtools

print("pysent", pysent.__version__, "from", Path(pysent.__file__).parent)

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. What is in the library

| Module | What it is for |
|---|---|
| `pysent.s1` / `pysent.s2` | the processing itself |
| `pysent.profiles` | detect S1 vs S2 from a name, URL or identifier |
| `pysent.archive` | catalogue download URL → local archive path; UUID → SAFE |
| `pysent.csw` | catalogue record lookup (extra: `csw`) |
| `pysent.qa` | benchmarking and stretch-quality measurement (extra: `qa`) |

Top-level names load lazily, so importing `pysent.profiles` does not require the
GDAL bindings that `pysent.s1` needs.

In [ ]:
for name in ("s1", "s2", "profiles", "archive", "csw", "qa"):
    module = getattr(pysent, name)
    first_line = (module.__doc__ or "").strip().split("\n")[0]
    print(f"pysent.{name:9s} {first_line}")

## 3. Platform detection\nDetection is pure string matching over any hints you have, so it works on a filename, a download URL or a catalogue title alike.

In [ ]:
from pysent.profiles import detect_nbs_sentinel_platform

for hint in [
    "S1A_IW_GRDH_1SDV_20260810T043022.zip",
    "S2C_MSIL2A_20260810T092031_N0512_R093_T35VPE.SAFE",
    "https://nbstds.met.no/.../nbsArchive/S1D/2026/08/10/IW/S1D_IW_GRDH_1SDV_....zip",
]:
    print(f"{detect_nbs_sentinel_platform(hint)}  <- {hint[:60]}")

## 4. Find something to process

In [ ]:
product = nbtools.resolve_input(
    PLATFORM,
    safe_path=SAFE_PATH or None,
    identifier=IDENTIFIER or None,
    data_root=DATA_ROOT or None,
    endpoint=ENDPOINT,
)
print(nbtools.describe(product))

## 5. Stretch it\nThis is the step that turns raw radiometry into something you can look at. Sentinel-2 counts span a wide range with a long bright tail (cloud, sunglint), so a **percentile** clip is used rather than min/max.

In [ ]:
from pysent.s2 import S2_STRETCH_PERCENTILES, stretch_sentinel_s2_rgb

with rasterio.open(product.path if product.is_fixture else product.path) as src:
    # Fixtures are already the 3 bands we want; a real product needs the warp
    # first (see 03_sentinel2.ipynb) - here we just read what is available.
    indexes = [1, 2, 3] if src.count >= 3 else [1]
    raw = src.read(indexes).astype(np.float32)
    nodata = src.nodata

print("raw:", raw.shape, raw.dtype, "range", raw.min(), "-", raw.max())

if raw.shape[0] == 3:
    stretched, stats = stretch_sentinel_s2_rgb(
        raw, nodata=nodata, percentiles=S2_STRETCH_PERCENTILES)
    print("percentiles used:", S2_STRETCH_PERCENTILES)
    for i, (lo, hi) in enumerate(zip(stats["p_low"], stats["p_high"])):
        print(f"  band {i + 1}: {lo:8.1f} -> 0   {hi:8.1f} -> 255")

## 6. Look at it\nLeft is the raw data at full min/max (**no** stretch applied) so the effect of the stretch on the right is actually visible. Displaying both stretched - the obvious mistake - makes them look identical and tells you nothing.

In [ ]:
import matplotlib.pyplot as plt

if raw.shape[0] == 3:
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    rgb_raw = np.transpose(raw, (1, 2, 0))
    axes[0].imshow((rgb_raw - rgb_raw.min()) / max(np.ptp(rgb_raw), 1))
    axes[0].set_title("raw (full min/max, unstretched)")
    axes[1].imshow(np.transpose(stretched, (1, 2, 0)))
    axes[1].set_title(f"percentile stretch {S2_STRETCH_PERCENTILES}")
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()

## 7. Where to go next\n\n- **[02_sentinel1.ipynb](02_sentinel1.ipynb)** - SAR: polarisation detection, the GCP warp, dB scaling.\n- **[03_sentinel2.ipynb](03_sentinel2.ipynb)** - optical: band combinations, min/max vs percentile.\n- **[04_benchmarks.ipynb](04_benchmarks.ipynb)** - time and memory per step; where the cost actually is.